In [ ]:
import os
from moviepy.editor import VideoFileClip

# --- SETTINGS ---
input_folder = "../data_source/videos"      # Folder containing your .mp4s
output_folder = "cropped_videos" # Where the new files will go
target_height = 720              # Your desired height in pixels

# Create output folder if it doesn't exist
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

# Loop through every file in the input folder
for filename in os.listdir(input_folder):
    if filename.endswith(".mp4"):
        input_path = os.path.join(input_folder, filename)
        output_path = os.path.join(output_folder, f"cropped_{filename}")
        
        print(f"Processing: {filename}...")
        
        try:
            with VideoFileClip(input_path) as video:
                # Calculate coordinates to center the crop vertically
                # If target_height is larger than the video, it skips cropping
                if video.h > target_height:
                    y_start = (video.h - target_height) / 2
                    y_end = y_start + target_height
                    
                    # Crop: keeping full width, centered height
                    final_clip = video.crop(y1=y_start, y2=y_end)
                else:
                    print(f"Skipping {filename}: Video height is already {video.h}px")
                    final_clip = video
                
                # Write file using multiple threads for speed on the cluster
                # match 'threads' to number of CPU cores requested
                final_clip.write_videofile(
                    output_path, 
                    codec="libx264", 
                    audio_codec="aac", 
                    threads=4, 
                    logger=None # Set to 'bar' if you want to see progress bars
                )
                
            print(f"Done: {output_path}")
            
        except Exception as e:
            print(f"Error processing {filename}: {e}")

print("--- All tasks complete! ---")